In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

**Note (provenance-only notebook):** this notebook reads raw per-session
recordings from an external drive path
(`/media/caveman/Vorchard1/kevin_split_flow_data/wildtypes/...`) that is not
part of this repository's `raw_data/` and is not published on Dryad. It
documents how the date-stamped `*_SHAM_AUGMENTED.parquet` files were produced
for the wildtype dataset. It is not expected to run on another machine and
does not need to be re-run to reproduce the figures.

In [2]:
from braid_analysis import braid_slicing

In [3]:
def get_preprocessed_file(directory, key='*_preprocessed.hdf'):
    files = glob.glob(os.path.join(directory, key))
    if len(files) == 0:
        raise FileNotFoundError(f'No _preprocessed.hdf file found in {directory}')
    if len(files) > 1:
        raise ValueError(f'Multiple _preprocessed.hdf files found in {directory}')
    return files[0]

In [4]:
dirname = '20260601_WT_675ms_MF_split-flow'
ZTHRESH = 0.35

In [5]:
fname = get_preprocessed_file('/media/caveman/Vorchard1/kevin_split_flow_data/wildtypes/' + dirname + '/preprocessed_data/')
df_flies = pd.read_hdf(fname)

fname = get_preprocessed_file('/media/caveman/Vorchard1/kevin_split_flow_data/wildtypes/' + dirname + '/preprocessed_data/', 'optotrigger_light_stimulus.hdf')
df_lights = pd.read_hdf(fname)

FileNotFoundError: No _preprocessed.hdf file found in /media/caveman/Vorchard1/kevin_split_flow_data/wildtypes/20260601_WT_675ms_MF_split-flow/preprocessed_data/

In [34]:
df_flies.keys()

Index(['obj_id', 'frame', 'timestamp', 'x', 'y', 'z', 'xvel', 'yvel', 'zvel',
       'P00', 'P01', 'P02', 'P11', 'P12', 'P22', 'P33', 'P44', 'P55',
       'obj_id_unique', 'speed_xy', 'course', 'course_smoothish',
       'ang_vel_smoothish'],
      dtype='object')

In [35]:
df_lights.keys()

Index(['frame', 'flash_frame', 'obj_id_triggered', 'trigger_exp',
       'trigger_description', 'lights_on_function', 'lights_on', 'duration',
       'intensity'],
      dtype='object')

# Get long objids 

In [36]:
long_objids = braid_slicing.get_long_obj_ids_fast_pandas(df_flies, obj_id_key='obj_id_unique', length=500)

In [37]:
df_flies = df_flies[df_flies.obj_id_unique.isin(long_objids)]

In [38]:
def get_trajectories_in_volume(df, zmin=0.1, zmax=0.4, ymin=-0.1, ymax=0.1, xmin=-0.3, xmax=0.3):
    in_volume = df[
        (df['x'] >= xmin) & (df['x'] <= xmax) &
        (df['y'] >= ymin) & (df['y'] <= ymax) &
        (df['z'] >= zmin) & (df['z'] <= zmax)
    ]
    return in_volume['obj_id_unique'].unique().tolist()

In [39]:
obj_ids_in_volume = get_trajectories_in_volume(df_flies)

In [40]:
df_flies = df_flies[df_flies.obj_id_unique.isin(obj_ids_in_volume)]

In [41]:
len(df_flies.obj_id_unique.unique())

38

In [42]:
def create_flash_aligned_trajectories(df_flies, zmin=0.07, zmax=0.4, ymin=-0.15, ymax=0.15, 
                                       xmin=-0.2, xmax=0.2, fps=100,
                                       frames_before=20, min_frames_after=300, max_frames_after=500):
    
    total_length = frames_before + 1 + max_frames_after  # 521 frames total
    result_dfs = []
    
    for obj_id, traj in df_flies.groupby('obj_id_unique'):
        traj = traj.sort_values('frame').reset_index(drop=True)
        
        # Find first frame inside volume
        in_volume = (
            (traj['x'] >= xmin) & (traj['x'] <= xmax) &
            (traj['y'] >= ymin) & (traj['y'] <= ymax) &
            (traj['z'] >= zmin) & (traj['z'] <= zmax)
        )
        
        if not in_volume.any():
            continue
        
        flash_idx = in_volume.idxmax()  # first True index
        
        # Check we have enough frames before
        if flash_idx < frames_before:
            continue
        
        # Slice: 20 before, flash frame, up to 500 after
        start_idx = flash_idx - frames_before
        end_idx = flash_idx + max_frames_after + 1  # +1 for inclusive slice
        
        frames_after = len(traj) - flash_idx - 1
        
        # Remove if fewer than min_frames_after
        if frames_after < min_frames_after:
            continue
        
        traj_trimmed = traj.iloc[start_idx:end_idx].copy().reset_index(drop=True)
        flash_frame_num = traj_trimmed.loc[frames_before, 'frame']
        
        # Add flash_frame bool column
        traj_trimmed['flash_frame'] = False
        traj_trimmed.loc[frames_before, 'flash_frame'] = True
        
        # Add time_relative_to_flash
        traj_trimmed['time_relative_to_flash'] = (traj_trimmed['frame'] - flash_frame_num) / fps
        
        # Pad with NaNs if shorter than total_length
        if len(traj_trimmed) < total_length:
            pad_length = total_length - len(traj_trimmed)
            last_frame = traj_trimmed['frame'].iloc[-1]
            pad_frames = pd.DataFrame({
                col: [np.nan] * pad_length for col in traj_trimmed.columns
            })
            # Fill in frame and time columns for padding
            pad_frame_nums = [last_frame + i + 1 for i in range(pad_length)]
            pad_frames['frame'] = pad_frame_nums
            pad_frames['time_relative_to_flash'] = [(f - flash_frame_num) / fps for f in pad_frame_nums]
            pad_frames['obj_id_unique'] = obj_id
            pad_frames['flash_frame'] = False
            traj_trimmed = pd.concat([traj_trimmed, pad_frames], ignore_index=True)
        
        result_dfs.append(traj_trimmed)
    
    if len(result_dfs) == 0:
        return pd.DataFrame()
    
    return pd.concat(result_dfs, ignore_index=True)

In [43]:
df_flies_aligned = create_flash_aligned_trajectories(df_flies)

In [44]:
df_flies_aligned

,obj_id,frame,timestamp,x,y,z,xvel,yvel,zvel,P00,...,P33,P44,P55,obj_id_unique,speed_xy,course,course_smoothish,ang_vel_smoothish,flash_frame,time_relative_to_flash
0,123.0,1519969,1.780373e+09,-0.224048,-0.009251,0.099772,0.234684,-0.109096,0.181635,0.000002,...,0.023951,0.024283,0.054477,20260601_164542_123,0.258802,-0.435143,-0.433798,12.518478,False,-0.20
1,123.0,1519970,1.780373e+09,-0.221454,-0.009717,0.101368,0.248990,-0.071507,0.172320,0.000002,...,0.023848,0.024195,0.053519,20260601_164542_123,0.259055,-0.279662,-0.278465,17.749933,False,-0.19
2,123.0,1519971,1.780373e+09,-0.218788,-0.009602,0.102899,0.259129,-0.021893,0.163174,0.000002,...,0.023829,0.024162,0.052697,20260601_164542_123,0.260053,-0.084286,-0.078800,20.091528,False,-0.18
3,123.0,1519972,1.780373e+09,-0.216041,-0.008873,0.103991,0.267537,0.035949,0.142284,0.000003,...,0.025528,0.025278,0.055686,20260601_164542_123,0.269941,0.133570,0.123366,18.495402,False,-0.17
4,123.0,1519973,1.780373e+09,-0.213678,-0.007815,0.105343,0.249820,0.075708,0.137724,0.000002,...,0.023854,0.024167,0.052178,20260601_164542_123,0.261040,0.294251,0.291108,15.697840,False,-0.16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9373,NaN,2205588,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,20260601_164542_946,NaN,NaN,NaN,NaN,False,4.96
9374,NaN,2205589,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,20260601_164542_946,NaN,NaN,NaN,NaN,False,4.97
9375,NaN,2205590,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,20260601_164542_946,NaN,NaN,NaN,NaN,False,4.98
9376,NaN,2205591,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,20260601_164542_946,NaN,NaN,NaN,NaN,False,4.99


In [45]:
len(df_flies_aligned.obj_id_unique.unique())

18

In [46]:
def get_trajectories_lights_off(df_flies, df_lights, n_frames=1000):
    result = []
    for obj_id, traj in df_flies.groupby('obj_id_unique'):
        first_frame = traj['frame'].min()
        last_frame = traj['frame'].max()
        
        # Check frames from 1000 before trajectory start through end of trajectory
        lights = df_lights[
            (df_lights['frame'] >= first_frame - n_frames) & 
            (df_lights['frame'] <= last_frame)
        ]['lights_on']
        
        if len(lights) > 0 and (lights == 0).all():
            result.append(obj_id)
    return result

In [47]:
obj_id_sham_candidates = get_trajectories_lights_off(df_flies_aligned, df_lights)

In [48]:
len(obj_id_sham_candidates)

12

In [49]:
df_flies_aligned_sham = df_flies_aligned[df_flies_aligned.obj_id_unique.isin(obj_id_sham_candidates)].copy()

In [62]:
def get_trajectories_below_z(df_flies, zmax=0.35):
    below_z = df_flies.groupby('obj_id_unique')['z'].max()
    return below_z[below_z < zmax].index.tolist()

In [63]:
obj_ids = get_trajectories_below_z(df_flies_aligned_sham)

In [64]:
len(obj_ids)

0

In [65]:
df_flies_aligned_sham = df_flies_aligned_sham[df_flies_aligned_sham.obj_id_unique.isin(obj_ids)].copy()

In [66]:
df_flies_aligned_sham.loc[:, 'obj_id_unique_event'] = df_flies_aligned_sham.obj_id_unique.values

In [67]:
if len(df_flies_aligned_sham.obj_id_unique_event.unique()) > 0:
        
    df_flies_aligned_sham.loc[:,'duration'] = 0
    df_flies_aligned_sham.loc[:,'intensity'] = 0
    df_flies_aligned_sham.loc[:,'trigger_descrition'] = 'augmented sham'
    df_flies_aligned_sham.loc[:,'lights_on_function'] = 'none'
    df_flies_aligned_sham.loc[:,'lights_on'] = 0

    df_flies_aligned_sham.to_parquet(dirname.split('_')[0] + '_' + 'flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed_SHAM_AUGMENTED.parquet')

    obj_id = df_flies_aligned_sham.obj_id_unique_event.unique()[0]
    trajec = df_flies_aligned_sham[df_flies_aligned_sham.obj_id_unique_event==obj_id]

    plt.plot(trajec.x, trajec.y)

# Merge individual parquets

In [28]:
def get_preprocessed_files(directory, key='*_preprocessed.hdf'):
    files = glob.glob(os.path.join(directory, key))
    return files

In [29]:
files = get_preprocessed_files('.', key='*flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed_SHAM_AUGMENTED.parquet')

In [30]:
files

['./20240513_flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed_SHAM_AUGMENTED.parquet',
 './20240516_flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed_SHAM_AUGMENTED.parquet',
 './20240111_flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed_SHAM_AUGMENTED.parquet',
 './20240514_flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed_SHAM_AUGMENTED.parquet',
 './20240110_flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed_SHAM_AUGMENTED.parquet',
 './20240109_flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed_SHAM_AUGMENTED.parquet']

In [31]:
dfs = []
for file in files:
    df = pd.read_parquet(file)
    dfs.append(df)

df_combined = pd.concat(dfs)

In [32]:
df_combined.to_parquet('../../../../../Data/Experimental_Fly_Data/flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed_SHAM_AUGMENTED_merged.parquet')

In [33]:
len(df_combined.obj_id_unique_event.unique())

86